# Harvest: box integration, cleaning, profile-fitted intensities and grain shapes
__Uses ImageD11.sinograms.harvester, profile_fit and harvest_pipeline__

From a scanning 3DXRD dataset and its indexed grains (the grains file and its TensorMap labels, from the tomo or pbp route), this integrates every predicted reflection of every grain on the raw frames, merges duplicate grains, flags reflections contaminated by other grains, learns the 1D peak profiles from strong reflections, harvests again with them as weights, and gives each grain a shape (MLEM) and per-reflection intensities.

The steps write to `outdir` and each one reads what the previous ones wrote, so they can be run one at a time. For a full dataset, run them as a batch job instead:

    python -m ImageD11.sinograms.harvest_pipeline outdir/config.json

Look at the results with `harvest_2_view.ipynb`.

In [ ]:
import os

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMBA_NUM_THREADS'] = str(max(1, len(os.sched_getaffinity(0)) // 2))

In [ ]:
# this cell is tagged with 'parameters'
# to view the tag, select the cell, then find the settings gear icon (right or left sidebar) and look for Cell Tags

# python environment stuff
IMAGED11_PATH = None  # means do not use git, otherwise "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = None  # None means guess, or you can specify a folder for the checkout

# dataset file and phase (with grains and a TensorMap_<phase> in the dataset's grains file)
dset_path = 'si_cube_test/processed/Si_cube/Si_cube_S3DXRD_nt_moves_dty/Si_cube_S3DXRD_nt_moves_dty_dataset.h5'
phase_str = 'Si'

# where the results go; raw_data = None: the dataset's own folder of scan files
outdir = os.path.join(os.path.dirname(os.path.abspath(dset_path)), 'harvest_' + phase_str)
raw_data = None
nproc = 8

# first harvest: reflections to |g| <= ds_max, box sizes from the ShoeboxModel (arguments; {} = defaults)
ds_max = 1.5
box = {}

# crystal symmetry for misorientations (duplicate grains)
symmetry = 'cubic'

# absorption: linear absorption coefficient per dty unit (None: no correction)
mu = None

# grain shapes from reflections with h, k, l all divisible by this (2: the fundamental reflections of a supercell)
hkl_divisor = 1

# which steps to run now (None: all)
steps = None

In [ ]:
if IMAGED11_PATH is not None:
    exec(open('/data/id11/nanoscope/install_ImageD11_from_git.py').read())
    PYTHONPATH=setup_ImageD11_from_git(CHECKOUT_PATH, IMAGED11_PATH)
else:
    import site
    PYTHONPATH = site.getsitepackages()[0]
    print(PYTHONPATH)

In [ ]:
import json
from ImageD11.sinograms import harvest_pipeline

# Config
Every policy value is in the config file written here (see `harvest_pipeline.DEFAULTS`). Edit it, or the dictionary below, to change one.

In [ ]:
if not os.path.isdir(outdir):
    os.makedirs(outdir)
config = dict(dataset=os.path.abspath(dset_path), phase=phase_str, outdir=outdir, raw_data=raw_data, nproc=nproc,
              harvest=dict(ds_max=ds_max, box=box), duplicates=dict(symmetry=symmetry),
              absorption=dict(mu=mu), mlem=dict(hkl_divisor=hkl_divisor))
config_file = os.path.join(outdir, 'config.json')
with open(config_file, 'w') as f:
    json.dump(config, f, indent=1)
cfg = harvest_pipeline.load_config(config_file)
print(json.dumps(cfg, indent=1))

# Run

In [ ]:
harvest_pipeline.run(cfg, steps)

# Summary

In [ ]:
import csv
with open(os.path.join(outdir, 'duplicates.json')) as f:
    print('duplicate groups (merged):', json.load(f)['groups'])
rows = list(csv.DictReader(open(os.path.join(outdir, 'grains.csv'))))
print(len(rows), 'grains,', sum(r['refine'] == 'True' for r in rows), 'kept for refinement')